# Check vs Wilson flux

Compares native interpolant fluxes with the raw occupied MMN Wilson flux. This is the downstream companion to `check_raw_mmn_wilson.ipynb`, which establishes that trials 03 and 04 have the same occupied projector, links, and Wilson plaquette phases on the original 8^3 mesh. Here, each trial's native MMN-derived, orbital-aware H(R)/A(R) interpolant is evaluated inside every original plaquette and

`integral_P Tr Omega_mn(k) dk_m dk_n`

is computed by tensor-product Gauss-Legendre quadrature. With the connection conventions used here, an occupied-overlap Wilson loop has phase `arg det W_mn = - integral_P Tr Omega_mn + finite-link corrections`. Two logically different residuals are reported:

1. `interpolant_vs_raw` compares `-integral Tr Omega` with the finite-link raw MMN Wilson phase -- combines finite-link/discretization error with any error from reconstructing a continuous interpolant from the 8^3 data.
2. `trial03_vs_trial04` compares the two continuum integrals under identical quadrature -- directly measures trial-dependent interpolation, does not treat the finite-link Wilson phase as continuum-exact.

All 512 plaquettes in xy, yz, zx are integrated for every requested order.

Set `RUN_COMPUTATION = False` to skip straight to loading the last saved run from `results/wilson_flux/`.

In [ ]:
import sys
from pathlib import Path

import csv
import gc
import json
import time
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation._paths import RESULTS
from validation.symmetry.compare_trial_curvature_lib import N_OCC, TRIALS, build_trial
from modules import curvature as cv
from wannier.wannier_io import read_wannier_checkpoint

## Config

In [ ]:
RUN_COMPUTATION = False  # True: build both native interpolants and recompute. False: load results/wilson_flux/.
ORDERS = (1, 2, 3, 4)  # Gauss-Legendre quadrature orders per plaquette
BATCH_SIZE = 256
OUTPUT_DIR = RESULTS / "wilson_flux"
RAW_DIR = RESULTS / "raw_mmn_wilson"
RAW_CSV = RAW_DIR / "raw_mmn_wilson_plaquettes.csv"
PLAQUETTES = ((0, 1, "xy"), (1, 2, "yz"), (2, 0, "zx"))
ORIENTATION_INDEX = {label: index for index, (_, _, label) in enumerate(PLAQUETTES)}
MESH_STEP = 1.0 / 8.0

## Helpers

In [ ]:
def wrap_phase(value: np.ndarray) -> np.ndarray:
    """Wrap real phases onto (-pi,pi]."""
    return np.angle(np.exp(1j * value))


def read_raw_flux() -> tuple[np.ndarray, np.ndarray]:
    """Read the raw polar-unitary Wilson flux and starting mesh points."""
    if not RAW_CSV.is_file():
        raise FileNotFoundError(
            f"{RAW_CSV} is missing; run the check_raw_mmn_wilson.ipynb notebook first"
        )
    raw_flux = np.full((3, 512), np.nan, dtype=float)
    kpoints = np.full((512, 3), np.nan, dtype=float)
    with RAW_CSV.open(newline="") as handle:
        for row in csv.DictReader(handle):
            orientation = ORIENTATION_INDEX[row["orientation"]]
            ik = int(row["k_index_zero_based"])
            raw_flux[orientation, ik] = float(row["trial03_polar_flux_rad"])
            point = np.array([float(row["kx_reduced"]), float(row["ky_reduced"]), float(row["kz_reduced"])])
            if np.isfinite(kpoints[ik]).all() and not np.allclose(kpoints[ik], point, atol=0, rtol=0):
                raise ValueError("inconsistent k point in raw plaquette CSV")
            kpoints[ik] = point
    if not np.isfinite(raw_flux).all() or not np.isfinite(kpoints).all():
        raise ValueError("raw plaquette CSV is incomplete")
    return raw_flux, kpoints


def quadrature_layout(kpoints: np.ndarray, order: int):
    """Return all quadrature nodes, weights, plane indices, and owners."""
    nodes, weights = np.polynomial.legendre.leggauss(order)
    nodes = 0.5 * (nodes + 1.0)
    weights = 0.5 * weights
    area_weights = MESH_STEP**2 * np.outer(weights, weights).reshape(-1)
    offsets = np.array([(left, right) for left in nodes for right in nodes], dtype=float)

    total = len(PLAQUETTES) * len(kpoints) * order**2
    points = np.empty((total, 3), dtype=float)
    point_weights = np.empty(total, dtype=float)
    orientation_indices = np.empty(total, dtype=int)
    owner_indices = np.empty(total, dtype=int)
    mu_indices = np.empty(total, dtype=int)
    nu_indices = np.empty(total, dtype=int)

    cursor = 0
    block = len(kpoints) * order**2
    for orientation, (mu, nu, _) in enumerate(PLAQUETTES):
        plane_points = np.repeat(kpoints, order**2, axis=0)
        tiled_offsets = np.tile(offsets, (len(kpoints), 1))
        plane_points[:, mu] += MESH_STEP * tiled_offsets[:, 0]
        plane_points[:, nu] += MESH_STEP * tiled_offsets[:, 1]
        plane_points %= 1.0
        section = slice(cursor, cursor + block)
        points[section] = plane_points
        point_weights[section] = np.tile(area_weights, len(kpoints))
        orientation_indices[section] = orientation
        owner_indices[section] = np.repeat(np.arange(len(kpoints)), order**2)
        mu_indices[section] = mu
        nu_indices[section] = nu
        cursor += block

    return points, point_weights, orientation_indices, owner_indices, mu_indices, nu_indices


def evaluate_trace_curvature(model, pos, kpoints: np.ndarray) -> np.ndarray:
    """Evaluate the three required reduced-coordinate traced curvatures."""
    curvature = cv.omega(cv.connection(cv.fields(model, pos, kpoints), N_OCC))
    traces = np.empty((3, len(kpoints)), dtype=float)
    for orientation, (mu, nu, _) in enumerate(PLAQUETTES):
        traces[orientation] = np.trace(curvature[mu, nu], axis1=-2, axis2=-1).real
    return traces


def integrate_trial(trial: str, orders, kpoints: np.ndarray, batch_size: int):
    """Build one native interpolant and integrate every requested plaquette."""
    print(f"building native {TRIALS[trial]['label']}", flush=True)
    model, pos = build_trial(trial)
    integrated = {}
    for order in orders:
        points, weights, orientations, owners, _, _ = quadrature_layout(kpoints, order)
        accumulator = np.zeros(3 * len(kpoints), dtype=float)
        start_time = time.perf_counter()
        print(f"  order {order}: {len(points)} quadrature points in batches of {batch_size}", flush=True)
        for start in range(0, len(points), batch_size):
            stop = min(start + batch_size, len(points))
            traces = evaluate_trace_curvature(model, pos, points[start:stop])
            local = np.arange(stop - start)
            values = traces[orientations[start:stop], local] * weights[start:stop]
            flat_owner = orientations[start:stop] * len(kpoints) + owners[start:stop]
            np.add.at(accumulator, flat_owner, values)
            if stop == len(points) or stop % max(2048, batch_size) == 0:
                print(f"    evaluated {stop}/{len(points)} points", flush=True)
        integrated[order] = accumulator.reshape(3, len(kpoints))
        print(f"  order {order} completed in {time.perf_counter() - start_time:.1f} s", flush=True)
    del model, pos
    gc.collect()
    return integrated


def norm_metrics(difference: np.ndarray, reference: np.ndarray) -> dict:
    """Return absolute and reference-scaled metrics for one orientation."""
    reference_norm = float(np.linalg.norm(reference))
    return {
        "maximum_absolute": float(np.max(np.abs(difference))),
        "rms": float(np.sqrt(np.mean(np.square(difference)))),
        "mean_absolute": float(np.mean(np.abs(difference))),
        "relative_l2": float(np.linalg.norm(difference) / reference_norm if reference_norm > 0 else np.nan),
    }


def build_summary(orders, raw_flux: np.ndarray, results: dict) -> dict:
    """Separate raw-link residuals, trial differences, and quadrature error."""
    summary = {
        "conventions": {
            "integrated_quantity": "I_mn = integral_plaquette Tr Omega_mn dk_m dk_n in reduced coordinates",
            "wilson_relation": "arg det W_mn = -I_mn plus finite-link/discretization corrections",
            "interpolant_vs_raw_residual": "wrap((-I_native) - phi_raw); this is not expected to vanish at finite 1/8 link length",
            "trial_residual": "I_trial04 - I_trial03 under identical quadrature",
        },
        "orders": {},
        "quadrature_convergence": {},
    }
    for order in orders:
        order_summary = {
            "points_per_plaquette": order**2,
            "total_quadrature_points_per_trial": 3 * 512 * order**2,
            "interpolant_vs_raw": {},
            "trial03_vs_trial04": {},
        }
        for trial in ("trial03", "trial04"):
            order_summary["interpolant_vs_raw"][trial] = {}
            predicted_wilson = wrap_phase(-results[trial][order])
            residual = wrap_phase(predicted_wilson - raw_flux)
            for orientation, (_, _, label) in enumerate(PLAQUETTES):
                order_summary["interpolant_vs_raw"][trial][label] = norm_metrics(residual[orientation], raw_flux[orientation])
        difference = results["trial04"][order] - results["trial03"][order]
        for orientation, (_, _, label) in enumerate(PLAQUETTES):
            order_summary["trial03_vs_trial04"][label] = norm_metrics(difference[orientation], results["trial03"][order][orientation])
        summary["orders"][str(order)] = order_summary

    highest = max(orders)
    for trial in ("trial03", "trial04"):
        summary["quadrature_convergence"][trial] = {}
        reference = results[trial][highest]
        for order in orders:
            summary["quadrature_convergence"][trial][str(order)] = {}
            difference = results[trial][order] - reference
            for orientation, (_, _, label) in enumerate(PLAQUETTES):
                summary["quadrature_convergence"][trial][str(order)][label] = norm_metrics(difference[orientation], reference[orientation])
    return summary


def save_wilson_results(output: Path, orders, kpoints, raw_flux, results, summary) -> None:
    """Save per-plaquette CSV, lossless arrays, and summary metrics."""
    output.mkdir(parents=True, exist_ok=True)
    csv_path = output / "interpolant_vs_raw_plaquette_flux.csv"
    npz_path = output / "interpolant_vs_raw_plaquette_flux.npz"
    json_path = output / "interpolant_vs_raw_plaquette_flux_summary.json"

    fields = [
        "quadrature_order", "orientation", "k_index_zero_based", "kx_reduced", "ky_reduced", "kz_reduced",
        "raw_wilson_flux_rad", "trial03_integral_trace_omega", "trial04_integral_trace_omega",
        "trial03_predicted_wilson_flux_rad", "trial04_predicted_wilson_flux_rad",
        "trial03_predicted_minus_raw_wrapped_rad", "trial04_predicted_minus_raw_wrapped_rad",
        "trial04_minus_trial03_integral_trace_omega",
    ]
    with csv_path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for order in orders:
            predicted03 = wrap_phase(-results["trial03"][order])
            predicted04 = wrap_phase(-results["trial04"][order])
            residual03 = wrap_phase(predicted03 - raw_flux)
            residual04 = wrap_phase(predicted04 - raw_flux)
            for orientation, (_, _, label) in enumerate(PLAQUETTES):
                for ik, kpoint in enumerate(kpoints):
                    writer.writerow({
                        "quadrature_order": order, "orientation": label, "k_index_zero_based": ik,
                        "kx_reduced": kpoint[0], "ky_reduced": kpoint[1], "kz_reduced": kpoint[2],
                        "raw_wilson_flux_rad": raw_flux[orientation, ik],
                        "trial03_integral_trace_omega": results["trial03"][order][orientation, ik],
                        "trial04_integral_trace_omega": results["trial04"][order][orientation, ik],
                        "trial03_predicted_wilson_flux_rad": predicted03[orientation, ik],
                        "trial04_predicted_wilson_flux_rad": predicted04[orientation, ik],
                        "trial03_predicted_minus_raw_wrapped_rad": residual03[orientation, ik],
                        "trial04_predicted_minus_raw_wrapped_rad": residual04[orientation, ik],
                        "trial04_minus_trial03_integral_trace_omega": (
                            results["trial04"][order][orientation, ik] - results["trial03"][order][orientation, ik]
                        ),
                    })

    arrays = {
        "orders": np.asarray(orders, dtype=int),
        "orientations": np.asarray([label for _, _, label in PLAQUETTES]),
        "kpoints_reduced": kpoints,
        "raw_wilson_flux_rad": raw_flux,
    }
    for trial in ("trial03", "trial04"):
        for order in orders:
            arrays[f"{trial}_integral_trace_omega_order{order}"] = results[trial][order]
    np.savez_compressed(npz_path, **arrays)
    json_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
    print(f"saved {csv_path}")
    print(f"saved {npz_path}")
    print(f"saved {json_path}")

## Read the raw Wilson flux and sanity-check the mesh

In [ ]:
if RUN_COMPUTATION:
    raw_flux, kpoints = read_raw_flux()
    checkpoint = read_wannier_checkpoint(Path(TRIALS["trial03"]["directory"]) / "SrTiO3.chk")
    if tuple(checkpoint["mp_grid"]) != (8, 8, 8):
        raise ValueError("this validation expects the original 8^3 mesh")
    if not np.allclose(checkpoint["kpt_red"], kpoints, atol=1e-13, rtol=0):
        raise ValueError("raw Wilson CSV and checkpoint k-point orders differ")

## Integrate both trials

`results[trial][order]` is the `(3, 512)` array of per-plaquette integrated trace curvature -- inspect it directly.

In [ ]:
if RUN_COMPUTATION:
    results = {trial: integrate_trial(trial, ORDERS, kpoints, BATCH_SIZE) for trial in ("trial03", "trial04")}

## Build the summary

In [ ]:
if RUN_COMPUTATION:
    summary = build_summary(ORDERS, raw_flux, results)
    summary["input"] = {
        "raw_wilson_csv": str(RAW_CSV),
        "mesh": [8, 8, 8],
        "plaquettes_per_orientation": 512,
        "orientations": [label for _, _, label in PLAQUETTES],
        "quadrature_orders": list(ORDERS),
        "batch_size": BATCH_SIZE,
        "position_source": "MMN-derived pair-distance A(R)",
        "H_and_A_mapping": "same orbital-dependent min|R+tau_right-tau_left| mapping",
    }
    print(json.dumps(summary, indent=2, sort_keys=True))

## Save

In [ ]:
if RUN_COMPUTATION:
    save_wilson_results(OUTPUT_DIR, ORDERS, kpoints, raw_flux, results, summary)

## Load previously saved results (skip if you just computed above)

In [ ]:
if not RUN_COMPUTATION:
    with np.load(OUTPUT_DIR / "interpolant_vs_raw_plaquette_flux.npz") as stored:
        ORDERS = tuple(int(o) for o in stored["orders"])
        kpoints = stored["kpoints_reduced"]
        raw_flux = stored["raw_wilson_flux_rad"]
        results = {
            trial: {order: stored[f"{trial}_integral_trace_omega_order{order}"] for order in ORDERS}
            for trial in ("trial03", "trial04")
        }
    summary = json.loads((OUTPUT_DIR / "interpolant_vs_raw_plaquette_flux_summary.json").read_text())
    print(json.dumps(summary["input"], indent=2))

## Plot: interpolant-vs-raw residual, highest order, all orientations

In [ ]:
highest = max(ORDERS)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), dpi=150, sharey=True)
for ax, (_, _, label) in zip(axes, PLAQUETTES):
    orientation = ORIENTATION_INDEX[label]
    predicted03 = np.angle(np.exp(1j * -results["trial03"][highest][orientation]))
    predicted04 = np.angle(np.exp(1j * -results["trial04"][highest][orientation]))
    residual03 = np.angle(np.exp(1j * (predicted03 - raw_flux[orientation])))
    residual04 = np.angle(np.exp(1j * (predicted04 - raw_flux[orientation])))
    ax.plot(residual03, label="trial03", linewidth=1)
    ax.plot(residual04, label="trial04", linewidth=1)
    ax.set_title(f"{label}, order {highest}")
    ax.set_xlabel("plaquette index")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("predicted - raw Wilson phase [rad]")
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()